In [2]:
import json, os
import pandas as pd
from jiwer import cer, wer


In [115]:
# with open(train_file, encoding="utf-8") as f:
#     sample = json.loads(next(f))

import json

train_file = "../icdar2017/en/hipe-ocrepair-bench_v0.9_icdar2017_v1.1_train_en.jsonl"
samples=[]
with open(train_file, encoding="utf-8") as f:
    for line in f:
        sample = json.loads(line)
        samples.append(sample) # print(sample)
sample

{'document_metadata': {'document_id': 'icdar2017_train_en_5_173',
  'primary_dataset_name': 'icdar2017',
  'primary_dataset_filename': 'data/03_curated/icdar2017/en/train/chunks/icdar2017__train__en__periodical__eng_periodical-5__chunk_173__GT.txt',
  'primary_dataset_version': 'v1.1',
  'primary_dataset_license': 'https://creativecommons.org/licenses/by-nc-sa/4.0/',
  'primary_dataset_url': 'https://sites.google.com/view/icdar2017-postcorrectionocr',
  'benchmark_dataset_name': 'hipe-ocrepair-bench-icdar2017',
  'benchmark_dataset_doi': 'https://doi.org/doi:10.5281/zenodo.18824428',
  'benchmark_dataset_split': 'train',
  'document_type': 'newspaper',
  'date': 'n/a',
  'language': 'en',
  'publication_title': 'icdar 2017 - en - periodical document 5 - chunk 173',
  'transcription_unit_scope': 'chunk',
  'segmentation_origin_article': 'primary-dataset - legacy',
  'segmentation_origin_sentences': 'benchmark - automatic',
  'segmentation_origin_paragraphs': 'benchmark - automatic'},
 '

In [ ]:
# train_file
sample=samples[0]
ocr_text = sample["ocr_hypothesis"]["transcription_unit"]
gt_text = sample["ground_truth"]["transcription_unit"]

print(f"OCR: {ocr_text}")
print(f"GT : {gt_text}")
print(len(ocr_text), len(ocr_text.split()))
print(len(gt_text),len(ocr_text.split()))
print("OCR CER :", cer(gt_text, ocr_text))
print("OCR WER :", wer(gt_text, ocr_text))

OCR: REDUCED FARES Settlers, by the New Zealand hipping Company's .38, Leadenhall street, London Steamers, rfrrther particulars and for all information Colony apply tothe Agent-General -r Zealand, 15, Victoria-Street, London Zealand Shipping Cos Agents-Escott, 35 Pans-street, Exeter Pinder .rkwell, 191, High-street H. J. Waring, . Millbay, Plymouth, a CSTRALIA, NEW ZBALAND7 -t TASMANIA. £NT LINE FORTNIGHTLY MAIL SERVICE. following Steamships leave LONDON as - PLYMOUTH one day later, NAPLES days later, tor ALBANY, ADELAIDE, f'i l' RN E, and SYDNEY, with calling at Colombo, and all Ports in Austral Asia. •d msh p Tons. H.-P. Date.
GT : REDUCED FARES Settlers, by the New Zealand Shipping Company's 38, Leadenhall Street, London Steamers. For further particulars and for all information Colony apply to the Agent-General for New Zealand, 13, Victoria-street, London Zealand Shipping Co's Agents-S A Escott, 35 Paris-street, Exeter Pinder and fuckwell, 191, High-street H. J. Waring, and Co., Mil

## repair one

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=
)

In [82]:
sample=samples[0]
ocr_text = sample["ocr_hypothesis"]["transcription_unit"]
gt_text = sample["ground_truth"]["transcription_unit"]

print(f"OCR: {ocr_text}")
print(f"GT : {gt_text}")
print(len(ocr_text), len(ocr_text.split()))
print(len(gt_text),len(ocr_text.split()))
print("OCR CER :", cer(gt_text, ocr_text))
print("OCR WER :", wer(gt_text, ocr_text))

OCR: REDUCED FARES Settlers, by the New Zealand hipping Company's .38, Leadenhall street, London Steamers, rfrrther particulars and for all information Colony apply tothe Agent-General -r Zealand, 15, Victoria-Street, London Zealand Shipping Cos Agents-Escott, 35 Pans-street, Exeter Pinder .rkwell, 191, High-street H. J. Waring, . Millbay, Plymouth, a CSTRALIA, NEW ZBALAND7 -t TASMANIA. £NT LINE FORTNIGHTLY MAIL SERVICE. following Steamships leave LONDON as - PLYMOUTH one day later, NAPLES days later, tor ALBANY, ADELAIDE, f'i l' RN E, and SYDNEY, with calling at Colombo, and all Ports in Austral Asia. •d msh p Tons. H.-P. Date.
GT : REDUCED FARES Settlers, by the New Zealand Shipping Company's 38, Leadenhall Street, London Steamers. For further particulars and for all information Colony apply to the Agent-General for New Zealand, 13, Victoria-street, London Zealand Shipping Co's Agents-S A Escott, 35 Paris-street, Exeter Pinder and fuckwell, 191, High-street H. J. Waring, and Co., Mil

In [ ]:
# 错误连接或者分开的单词，拼写错误，
# ocr噪音导致的在数字或单词中间的标点
#【不要】修正语法，增加新词，moderniser单词 (and => &), 改变不需要改变的标点，去掉内容，改变专有名词和大小写


def correct_by_llm(ocr_text, model="google/gemini-3-flash-preview"):
    SYSTEM_PROMPT="""
    You are an OCR correction system.

    Correct obvious OCR recognition errors only.

    Allowed corrections:
    - fix misspelled words caused by OCR errors
    - split incorrectly merged words
    - merge incorrectly split words
    - remove punctuation that is clearly an OCR artifact inside a word or number

    DO NOT:
    - rewrite text
    - correct grammar
    - add any new word
    - add any new punctuation
    - replace word by punctuation
    - change regular punctuation
    - remove content
    - modernize language
    - change names, numbers, capitalization, and formatting

    If uncertain, keep the original text.

    Output only the corrected text.

    """

    USER_PROMPT=f"""
    Correct OCR errors in the following newspaper text.

    OCR TEXT:

    {ocr_text}
    """
     
    messages=[
        {'role':"system","content":SYSTEM_PROMPT},
        {'role':"user","content":USER_PROMPT}
    ]

    responses=client.chat.completions.create(
        model=model,
        temperature=0,# préférer la stabilité pour les textes histo!
        seed=42, 
        messages=messages,
    )

    prediction=responses.choices[0].message.content

    return prediction

prediction=correct_by_llm(ocr_text, model="google/gemini-3-flash-preview")

In [87]:
# check 
print('RAW:\n',ocr_text)
print('GT:\n', gt_text)
print('PRED:\n',prediction)


RAW:
 REDUCED FARES Settlers, by the New Zealand hipping Company's .38, Leadenhall street, London Steamers, rfrrther particulars and for all information Colony apply tothe Agent-General -r Zealand, 15, Victoria-Street, London Zealand Shipping Cos Agents-Escott, 35 Pans-street, Exeter Pinder .rkwell, 191, High-street H. J. Waring, . Millbay, Plymouth, a CSTRALIA, NEW ZBALAND7 -t TASMANIA. £NT LINE FORTNIGHTLY MAIL SERVICE. following Steamships leave LONDON as - PLYMOUTH one day later, NAPLES days later, tor ALBANY, ADELAIDE, f'i l' RN E, and SYDNEY, with calling at Colombo, and all Ports in Austral Asia. •d msh p Tons. H.-P. Date.
GT:
 REDUCED FARES Settlers, by the New Zealand Shipping Company's 38, Leadenhall Street, London Steamers. For further particulars and for all information Colony apply to the Agent-General for New Zealand, 13, Victoria-street, London Zealand Shipping Co's Agents-S A Escott, 35 Paris-street, Exeter Pinder and fuckwell, 191, High-street H. J. Waring, and Co., Mi

In [88]:
ocr_cer = cer(gt_text, ocr_text)
model_cer = cer(gt_text, prediction)
improvement_cer = (ocr_cer - model_cer) / ocr_cer
print(f"original cer:{ocr_cer}")
print(f"repaired cer:{model_cer}")
print(f"cer delta : {model_cer-ocr_cer}")
# print(f"improuved: {model_cer<ocr_cer}, degraded : {model_cer> ocr_cer}")
print(f"improvement cer:{improvement_cer} \n")

ocr_wer = wer(gt_text, ocr_text)
model_wer = wer(gt_text, prediction)
improvement_wer = (ocr_wer - model_wer) / ocr_wer

print(f"original wer:{ocr_wer}")
print(f"repaired wer:{model_wer}")
print(f"wer delta : {model_wer-ocr_wer}")
print(f"improvement wer:{improvement_wer}")

original cer:0.11094452773613193
repaired cer:0.053973013493253376
cer delta : -0.05697151424287856
improvement cer:0.5135135135135135 

original wer:0.3711340206185567
repaired wer:0.21649484536082475
wer delta : -0.15463917525773196
improvement wer:0.4166666666666667


## pipeline

In [105]:
len(samples)

455

In [120]:
from tqdm import tqdm
import time

start_time=time.time()

results=[]
for i, sample in enumerate(tqdm(samples, desc='reparing OCR...')):

    # print(sample)
    document_id=sample['document_metadata']['document_id']
    ocr_text = sample["ocr_hypothesis"]["transcription_unit"]
    gt_text = sample["ground_truth"]["transcription_unit"]
    
    # correction :
    prediction=correct_by_llm(ocr_text)

    # eval
    ocr_cer = cer(gt_text, ocr_text)
    model_cer = cer(gt_text, prediction)
    delta_cer= model_cer-ocr_cer
    improvement_cer = (ocr_cer - model_cer) / ocr_cer
    
    ocr_wer = wer(gt_text, ocr_text)
    model_wer = wer(gt_text, prediction)
    delta_wer= model_wer-ocr_wer
    improvement_wer = (ocr_wer - model_wer) / ocr_wer

    res={
        'document_id':document_id,
        'ocr_text':ocr_text,
        'gt_text':gt_text,
        'repaired_text':prediction,

        'ocr_cer':ocr_cer,
        'model_cer':model_cer,
        "delta_cer":delta_cer,
        'improvement_cer':improvement_cer,
        
        'ocr_wer':ocr_wer,
        'model_wer':model_wer,
        "delta_wer":delta_wer,
        'improvement_wer':improvement_wer
    }
    results.append(res)
    # break 
end_time=time.time()
print(f"RUNTIME : {end_time-start_time:.2f} sec on {i+1} texts.\n")
print(results)


reparing OCR...: 100%|██████████| 455/455 [43:02<00:00,  5.68s/it]    

RUNTIME : 2582.23 sec on 455 texts.

[{'document_id': 'icdar2017_train_en_58_93', 'ocr_text': "REDUCED FARES Settlers, by the New Zealand hipping Company's .38, Leadenhall street, London Steamers, rfrrther particulars and for all information Colony apply tothe Agent-General -r Zealand, 15, Victoria-Street, London Zealand Shipping Cos Agents-Escott, 35 Pans-street, Exeter Pinder .rkwell, 191, High-street H. J. Waring, . Millbay, Plymouth, a CSTRALIA, NEW ZBALAND7 -t TASMANIA. £NT LINE FORTNIGHTLY MAIL SERVICE. following Steamships leave LONDON as - PLYMOUTH one day later, NAPLES days later, tor ALBANY, ADELAIDE, f'i l' RN E, and SYDNEY, with calling at Colombo, and all Ports in Austral Asia. •d msh p Tons. H.-P. Date.", 'gt_text': "REDUCED FARES Settlers, by the New Zealand Shipping Company's 38, Leadenhall Street, London Steamers. For further particulars and for all information Colony apply to the Agent-General for New Zealand, 13, Victoria-street, London Zealand Shipping Co's Agents-S

In [ ]:
import json, os
path_results='../results/repair_results_train.json'
# os.makedirs(os.path.dirname(path_results), exist_ok=True)
# with open(path_results, 'w', encoding='utf-8') as f:
#     json.dump(results, f, indent=2, ensure_ascii=False)

with open(path_results, 'r', encoding='utf-8') as f:
    results =json.load(f)
results[0]

{'document_id': 'icdar2017_train_en_58_93',
 'ocr_text': "REDUCED FARES Settlers, by the New Zealand hipping Company's .38, Leadenhall street, London Steamers, rfrrther particulars and for all information Colony apply tothe Agent-General -r Zealand, 15, Victoria-Street, London Zealand Shipping Cos Agents-Escott, 35 Pans-street, Exeter Pinder .rkwell, 191, High-street H. J. Waring, . Millbay, Plymouth, a CSTRALIA, NEW ZBALAND7 -t TASMANIA. £NT LINE FORTNIGHTLY MAIL SERVICE. following Steamships leave LONDON as - PLYMOUTH one day later, NAPLES days later, tor ALBANY, ADELAIDE, f'i l' RN E, and SYDNEY, with calling at Colombo, and all Ports in Austral Asia. •d msh p Tons. H.-P. Date.",
 'gt_text': "REDUCED FARES Settlers, by the New Zealand Shipping Company's 38, Leadenhall Street, London Steamers. For further particulars and for all information Colony apply to the Agent-General for New Zealand, 13, Victoria-street, London Zealand Shipping Co's Agents-S A Escott, 35 Paris-street, Exeter P

## eval:

In [7]:
import numpy as np

avg_ocr_cer = np.mean([r["ocr_cer"] for r in results])
avg_model_cer = np.mean([r["model_cer"] for r in results])

improvement = (avg_ocr_cer - avg_model_cer) / avg_ocr_cer

print("OCR CER:", avg_ocr_cer)
print("MODEL CER:", avg_model_cer)
print("IMPROVEMENT:", improvement)

OCR CER: 0.04305006240249153
MODEL CER: 0.02082081641784752
IMPROVEMENT: 0.516358043266332


In [9]:
avg_ocr_cer = np.mean([r["ocr_wer"] for r in results])
avg_model_cer = np.mean([r["model_wer"] for r in results])

improvement = (avg_ocr_cer - avg_model_cer) / avg_ocr_cer

print("OCR WER:", avg_ocr_cer)
print("MODEL WER:", avg_model_cer)
print("IMPROVEMENT:", improvement)

OCR WER: 0.18005717751632944
MODEL WER: 0.07543600911353497
IMPROVEMENT: 0.5810441430101076
